# Paper-Ready Table Exports

This notebook rebuilds the manuscript tables from generated repo outputs and writes paste-ready Markdown, Word, and LaTeX files. It does not change the analysis panel.


## Setup

Run this notebook from the repository root or from the `notebooks/` directory. The code adds `src/` to the Python path so the local package is used.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

# Find the repository folder whether this notebook is opened from the root or notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Tell Python to use the local project code instead of any installed package with the same name.
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# These functions rebuild the export files from the current analysis outputs.
from coa_finaid_subs.descstat_tables import build_descstat_tables, format_export_table
from coa_finaid_subs.estimate_tables import build_estimate_tables

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
REPO


## Table 1: Descriptive Statistics

This table is built from the prepared public/private nonprofit analysis panel. Winsorization is display-only and follows `config/descstat_variables.csv`.


In [ ]:
# The descriptive table starts from the prepared public/private nonprofit analysis panel.
PANEL = REPO / "outputs" / "analysis_panel" / "public_private_nonprofit" / "analysis_panel_coa_headroom_2009_2023_public_private_nonprofit.parquet"

# The config file controls which variables appear and which display-only caps apply.
DESC_CONFIG = REPO / "config" / "descstat_variables.csv"
DESC_OUTPUT = REPO / "outputs" / "descriptive_tables"

# This writes CSV, Markdown, LaTeX, and Word versions of the descriptive tables.
desc_paths = build_descstat_tables(
    input_panel=PANEL,
    output_dir=DESC_OUTPUT,
    config=DESC_CONFIG,
    scope_label="public_private_nonprofit",
)
desc_paths


In [ ]:
# Load the short paper table and format numbers for screen review.
desc_paper = pd.read_csv(desc_paths["paper_csv"])
display(format_export_table(desc_paper))


In [ ]:
# Show the same table as a paste-ready Markdown preview.
display(Markdown(desc_paths["paper_md"].read_text(encoding="utf-8")))


## Appendix Descriptive Table

The appendix version keeps missingness, tail quantiles, winsorization caps, and capped-row counts.


In [ ]:
# Load the longer appendix table with missingness, tails, and cap details.
desc_appendix = pd.read_csv(desc_paths["appendix_csv"])
display(format_export_table(desc_appendix))


## Table 2: Fixed-Effects Estimates

This table is rebuilt from `outputs/fixed_effects/`. It reports the configured focal estimates and selected component or interaction terms used in the current paper draft.


In [ ]:
# The fixed-effects table starts from generated estimates, not hand-entered numbers.
FE_DIR = REPO / "outputs" / "fixed_effects"
EST_OUTPUT = REPO / "outputs" / "estimate_tables"

# This writes CSV, Markdown, LaTeX, and Word versions of the estimate table.
estimate_paths = build_estimate_tables(
    fixed_effects_dir=FE_DIR,
    output_dir=EST_OUTPUT,
)
estimate_paths


In [ ]:
# Load the estimate table exactly as it was exported.
estimate_table = pd.read_csv(estimate_paths["paper_csv"])
display(estimate_table)


In [ ]:
# Show the estimate table as a paste-ready Markdown preview.
display(Markdown(estimate_paths["paper_md"].read_text(encoding="utf-8")))


## Export Paths

Use the Word files for direct copy/paste into Word or Google Docs. Use the LaTeX files in a manuscript with `booktabs` and `longtable` loaded. The Markdown files are plain paste-ready previews.


In [ ]:
# Collect the export paths so they are easy to find after the notebook runs.
export_rows = []
for label, paths in [("descriptive", desc_paths), ("fixed_effects", estimate_paths)]:
    for key, path in paths.items():
        if key.endswith(("_md", "_tex", "_docx", "_csv")):
            export_rows.append({"table_set": label, "format": key, "path": str(path)})
exports = pd.DataFrame(export_rows)
display(exports)
